In [1]:
from langchain_core.messages import HumanMessage
from simple_workflows import *
from simple_tools import *
from workflows_as_tools import *
from dotenv import load_dotenv
import os
load_dotenv()


True

In [2]:
### This tool takes two version of the same file and creates a new one that actually has the best of both worlds
### As it is, it just fixes an issue with citation format with nougat by leveraging the ocr one gets from mupdf
###(which is lower quality but with the right format). With a little bit of prompting tweeking it can get more
### general tasks of this nature. This tool needs an embeding mechanism as well. The reason is to locate the pages
### in the second file that correspond to the pages in the first.
from PIL import Image
ocr_app=OcrWorkflow()
ocr_app=ocr_app.create_workflow()
ocr_app=ocr_app.compile()
input={"main_text_filename": HumanMessage(content="Li")}
state=ocr_app.invoke(input)    


ocr in progress


 50%|█████     | 3/6 [01:00<01:00, 20.00s/it]Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 4.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..
Retrying langchain_g

keyword_and_summary completed successfully and the resulted file is named Li_keyword_and_summary


In [ ]:
### Same as before but in a form of a tool. Just for testing purposes.

OcrEnhancingTool=OcrEnhancingToolClass()
OcrEnhancingTool=StructuredTool(name="OcrEnhancingTool",func=OcrEnhancingTool.ocr_enhance,args_schema=OcrEnhancingInput,
                            description=OcrEnhancingTool.description)
OcrEnhancingTool.invoke({"main_text_filename": "Li", "supporting_text_filename": "mu_Li"})

In [7]:
### The idea of this chain/tool is to remove the proofs from a paper so it will be easier to make a summary out of it. 
### The tool uses two chains under the hood. The first stamps the pages of the text that continue a proof from the previous page.
### The idea was to help the LLM a bit to recognize proofs. The second LLM is doint the removal.
### From all the modules I created, this was the most unsucessful one. Even with strong LLMs like GPT-4o and Opus, I had partial results.
### I welcome anyone who can improve the prompt for this tool.
proof_remover_app=ProofRemovingWorkflow()
proof_remover_app=proof_remover_app.create_workflow()
proof_remover_app=proof_remover_app.compile()
input={"main_text_filename": HumanMessage(content="Li"),"file":[""],"report":HumanMessage(content="")}
state=proof_remover_app.invoke(input)


Stamping phase is initiated.


  0%|          | 0/34 [02:54<?, ?it/s]


ValueError: Your location is not supported by google-generativeai at the moment. Try to use ChatVertexAI LLM from langchain_google_vertexai.

In [ ]:
### Same as before but in a form of a tool. Just for testing purposes.

ProofRemoverTool=ProofRemovalToolClass()
ProofRemoverTool=StructuredTool(name="ProofRemovalTool",func=ProofRemoverTool.remove_proof,args_schema=ProofRemovalInput,
                            description=ProofRemoverTool.description)
ProofRemoverTool.invoke({"main_text_filename": "Li"})

In [ ]:
### This tool takes a text found in the folder files/markdowns and creates a set of keywords and a summary. in a form of a string and extracts the keywords and summary.
### It is preferable to use a file that doesnt contain proofs because it produces a better summary. 
input={"main_text_filename": HumanMessage(content="Li"),
           "report":HumanMessage(content=""),}
keyword_and_summary_app=KeywordAndSummaryWorkflow()
keyword_and_summary_app=keyword_and_summary_app.create_workflow()
keyword_and_summary_app=keyword_and_summary_app.compile()
state=keyword_and_summary_app.invoke(input)

In [ ]:
### Same as before but in a form of a tool. Just for testing purposes.

KeywordAndSummaryTool=KeywordAndSummaryToolClass()
KeywordAndSummaryTool=StructuredTool(name="KeywordAndSummaryTool",func=KeywordAndSummaryTool.get_keyword_and_summary,
                                         args_schema=KeywordSummaryCreationInput, description=KeywordAndSummaryTool.description)
KeywordAndSummaryTool.invoke({"main_text_filename": "Li"})

In [ ]:
### This workflow trnaslates the text found in the folder files/markdowns/main_text_filename  to the target language.
### it uses auxilary file for context based translation

input={"auxilary_text_filename": HumanMessage(content="Li_keyword_and_summary"), "target_language":HumanMessage(content="en"),"main_text_filename": HumanMessage(content="Li"), "report":HumanMessage}

translation_app=TranslationWorkflow()
translation_app=translation_app.create_workflow()
translation_app=translation_app.compile()
state=translation_app.invoke(input)
print(state)

In [ ]:
TranslationTool =TranslationToolClass()
    
TranslationTool=StructuredTool(name="TranslationTool",func=TranslationTool.translate_file,args_schema=TranslationInput,description=TranslationTool.description)
TranslationTool.invoke(input={"auxilary_text_filename":"Li_keyword_and_summary","target_language":"en","main_text_filename":"Li"})

In [ ]:
### This workflow takes a text found in the folder files/markdowns and creates a set of citations.
### Extraction type should be provided (all of them/ most important/ related to {topic}/female_authors)
### An auxilary text can be provided which will help judge better if the extraction type matches.

input={"auxilary_text_filename": HumanMessage(content="Li_keyword_and_summary"), "extraction_type":HumanMessage(content="All of them"),"main_text_filename": HumanMessage(content="Li"), 
"report":HumanMessage(content="")}

citation_extraction_app=CitationExtractionWorkflow()
citation_extraction_app=citation_extraction_app.create_workflow()
citation_extraction_app=citation_extraction_app.compile()
state=citation_extraction_app.invoke(input)
print(state)

In [ ]:
CitationExtractionTool=CitationExtractionToolClass()

CitationExtractionTool=StructuredTool(name="CitationExtractionTool",func=CitationExtractionTool.extract_citations,args_schema=CitationExtractionInput,
                           description=CitationExtractionTool.description)  

CitationExtractionTool.invoke(input={"main_text_filename":"Li","extraction_type":"All that appear in pages with proofs","auxilary_text_filename":"Li_keyword_and_summary"})

In [ ]:
### It skims a ttext and bring a very quick report. This will be only used as a tool for the manager, so it will not
### request a tool usage that corresponds to a heavy workflow if this is not necceessary. It will take a peak on the file and get an idea.
### Also useful for take citation files and sending them to the arxiv extractor.
input={"main_text_filename": HumanMessage(content="Li"),"report":HumanMessage(content="")}
take_a_peak_app=TakeAPeakWorkflow()
take_a_peak_app=take_a_peak_app.create_workflow()
take_a_peak_app=take_a_peak_app.compile()
state=take_a_peak_app.invoke(input)
print(state)

In [ ]:
TakeAPeakTool=TakeAPeakToolClass()

TakeAPeakTool=StructuredTool(name="TakeAPeakTool",func=TakeAPeakTool.take_a_peak,args_schema=TakeAPeakInput,
                           description=TakeAPeakTool.description)   

TakeAPeakTool.invoke(input={"main_text_filename":"Li"})
